# 00 — Collect OSS Repo Health Data (GitHub GraphQL API)

This notebook pulls activity/health signals for a stratified sample of GitHub
repos, using the GraphQL API (one query per repo covers commits, issues, PRs,
and contributors — much cheaper on rate limits than separate REST calls).

**Before running:** set your `GITHUB_TOKEN` as an environment variable
(see the setup steps you were given) — do NOT paste your token into any
cell below. If it's not set, the next cell will prompt you to enter it
just for this session (not saved to disk or to this notebook file).


In [2]:
import os
import json
import time
from datetime import datetime, timezone, timedelta
from getpass import getpass

import requests

TOKEN = os.environ.get("GITHUB_TOKEN")
if not TOKEN:
    print("GITHUB_TOKEN not found in environment.")
    TOKEN = getpass("Paste your GitHub token (input hidden): ").strip()

GRAPHQL_URL = "https://api.github.com/graphql"
SEARCH_URL = "https://api.github.com/search/repositories"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}

# quick check that the token works and see your rate limit
resp = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
resp.raise_for_status()
limits = resp.json()["resources"]
print("GraphQL remaining:", limits["graphql"]["remaining"], "/", limits["graphql"]["limit"])
print("Search remaining:", limits["search"]["remaining"], "/", limits["search"]["limit"])

GITHUB_TOKEN not found in environment.


Paste your GitHub token (input hidden):  ········


GraphQL remaining: 5000 / 5000
Search remaining: 30 / 30


## Configuration

Star tiers to sample from, so the dataset includes both thriving projects
and quiet/abandoned ones — not just mega-popular repos (which are almost
always "healthy" and would make for a trivial, imbalanced label).

In [5]:
STAR_TIERS = [
    ("mega", "stars:>50000"),
    ("large", "stars:10000..50000"),
    ("mid", "stars:1000..10000"),
    ("small", "stars:100..1000"),
    ("tiny", "stars:10..100"),
]

REPOS_PER_TIER = 100   # 100 x 5 tiers = 500 repos total
OUT_PATH = "../data/repo_health_raw.json"

os.makedirs("../data", exist_ok=True)

## GraphQL query

One request per repo pulls: basic metadata, issue/PR totals, the 50 most
recent issues and PRs (to compute response/merge times), and commit activity
via `since`-filtered counts (90-day and 365-day windows) rather than
averaging over a fixed number of nodes — a fixed-node average blows up when
commits land in bursts on a single day.

In [6]:
REPO_QUERY = """
query($owner: String!, $name: String!, $since90: GitTimestamp!, $since365: GitTimestamp!) {
  repository(owner: $owner, name: $name) {
    nameWithOwner
    stargazerCount
    forkCount
    createdAt
    pushedAt
    isArchived
    primaryLanguage { name }
    licenseInfo { key }
    openIssues: issues(states: OPEN) { totalCount }
    closedIssues: issues(states: CLOSED) { totalCount }
    recentIssues: issues(first: 50, orderBy: {field: CREATED_AT, direction: DESC}) {
      nodes { createdAt closedAt state }
    }
    openPRs: pullRequests(states: OPEN) { totalCount }
    mergedPRs: pullRequests(states: MERGED) { totalCount }
    recentPRs: pullRequests(first: 50, orderBy: {field: CREATED_AT, direction: DESC}) {
      nodes { createdAt mergedAt closedAt state }
    }
    defaultBranchRef {
      target {
        ... on Commit {
          lastCommit: history(first: 1) {
            nodes { committedDate }
          }
          commits90d: history(since: $since90) {
            totalCount
          }
          commits365d: history(since: $since365) {
            totalCount
          }
          recentAuthors: history(first: 100) {
            nodes { author { user { login } } }
          }
        }
      }
    }
    mentionableUsers { totalCount }
  }
}
"""

## Helper functions

`search_repos` — finds candidate repos per star tier. Deliberately sorted by
**stars**, not "recently updated" — sorting by recency would only return
repos that are currently active, which would bias the whole dataset toward
"healthy" and defeat the point of finding dormant/at-risk ones.

`fetch_repo_detail` — runs the GraphQL query for one repo, with light retry
on transient GitHub 502/503 errors.

`flatten` — turns the nested GraphQL response into one flat row, and derives
signals like `days_since_last_commit`, `commits_per_month_90d`, issue
resolution time, and PR merge time/rate.

In [7]:
def search_repos(query, target_count, per_tier_page_limit=5):
    results = []
    page = 1
    while len(results) < target_count and page <= per_tier_page_limit:
        resp = requests.get(
            SEARCH_URL,
            headers=HEADERS,
            params={"q": query, "sort": "stars", "order": "desc",
                    "per_page": 100, "page": page},
        )
        if resp.status_code != 200:
            print(f"  search error {resp.status_code}: {resp.text[:200]}")
            break
        items = resp.json().get("items", [])
        if not items:
            break
        results.extend(items)
        page += 1
        time.sleep(1)  # search endpoint has its own tighter rate limit (30/min)
    return results[:target_count]

In [8]:
def fetch_repo_detail(owner, name, max_retries=3):
    now = datetime.now(timezone.utc)
    since90 = (now - timedelta(days=90)).strftime("%Y-%m-%dT%H:%M:%SZ")
    since365 = (now - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")
    variables = {"owner": owner, "name": name, "since90": since90, "since365": since365}

    for attempt in range(max_retries):
        resp = requests.post(GRAPHQL_URL, headers=HEADERS,
                              json={"query": REPO_QUERY, "variables": variables})
        if resp.status_code == 200:
            body = resp.json()
            if "errors" in body and not body.get("data", {}).get("repository"):
                return None
            return body.get("data", {}).get("repository")
        elif resp.status_code in (502, 503):
            time.sleep(2 ** attempt)
            continue
        else:
            print(f"  graphql error {resp.status_code} for {owner}/{name}: {resp.text[:200]}")
            return None
    return None

In [9]:
def flatten(repo):
    if repo is None:
        return None

    now = datetime.now(timezone.utc)

    def parse(ts):
        return datetime.fromisoformat(ts.replace("Z", "+00:00")) if ts else None

    pushed_at = parse(repo["pushedAt"])
    created_at = parse(repo["createdAt"])
    days_since_push = (now - pushed_at).days if pushed_at else None
    age_days = (now - created_at).days if created_at else None

    branch = repo.get("defaultBranchRef") or {}
    target = branch.get("target") or {}

    last_commit_nodes = (target.get("lastCommit") or {}).get("nodes", [])
    days_since_last_commit = None
    if last_commit_nodes and last_commit_nodes[0].get("committedDate"):
        days_since_last_commit = (now - parse(last_commit_nodes[0]["committedDate"])).days

    commits_90d = (target.get("commits90d") or {}).get("totalCount", 0)
    commits_365d = (target.get("commits365d") or {}).get("totalCount", 0)
    commits_per_month_90d = round(commits_90d / 3, 2)
    commits_per_month_365d = round(commits_365d / 12, 2)

    recent_author_nodes = (target.get("recentAuthors") or {}).get("nodes", [])
    commit_authors = {
        (c.get("author") or {}).get("user", {}).get("login")
        for c in recent_author_nodes if (c.get("author") or {}).get("user")
    }
    commit_authors.discard(None)

    issue_nodes = repo["recentIssues"]["nodes"]
    closed_issues_recent = [i for i in issue_nodes if i["state"] == "CLOSED" and i["closedAt"]]
    resolution_days = []
    for i in closed_issues_recent:
        c, cl = parse(i["createdAt"]), parse(i["closedAt"])
        if c and cl:
            resolution_days.append((cl - c).days)
    median_issue_resolution_days = (
        sorted(resolution_days)[len(resolution_days) // 2] if resolution_days else None
    )

    pr_nodes = repo["recentPRs"]["nodes"]
    merged_prs_recent = [p for p in pr_nodes if p["state"] == "MERGED" and p["mergedAt"]]
    pr_merge_days = []
    for p in merged_prs_recent:
        c, m = parse(p["createdAt"]), parse(p["mergedAt"])
        if c and m:
            pr_merge_days.append((m - c).days)
    median_pr_merge_days = sorted(pr_merge_days)[len(pr_merge_days) // 2] if pr_merge_days else None
    pr_merge_rate_recent = round(len(merged_prs_recent) / len(pr_nodes), 2) if pr_nodes else None

    return {
        "repo": repo["nameWithOwner"],
        "stars": repo["stargazerCount"],
        "forks": repo["forkCount"],
        "language": (repo.get("primaryLanguage") or {}).get("name"),
        "license": (repo.get("licenseInfo") or {}).get("key"),
        "is_archived": repo["isArchived"],
        "age_days": age_days,
        "days_since_last_push": days_since_push,
        "days_since_last_commit": days_since_last_commit,
        "commits_per_month_90d": commits_per_month_90d,
        "commits_per_month_365d": commits_per_month_365d,
        "distinct_recent_commit_authors": len(commit_authors),
        "total_mentionable_users": repo["mentionableUsers"]["totalCount"],
        "open_issues_total": repo["openIssues"]["totalCount"],
        "closed_issues_total": repo["closedIssues"]["totalCount"],
        "median_issue_resolution_days_recent": median_issue_resolution_days,
        "open_prs_total": repo["openPRs"]["totalCount"],
        "merged_prs_total": repo["mergedPRs"]["totalCount"],
        "median_pr_merge_days_recent": median_pr_merge_days,
        "pr_merge_rate_recent": pr_merge_rate_recent,
    }

## Collection loop (with checkpointing)

Saves progress to disk after **every tier**, and skips tiers already
completed if you rerun this cell — so if it errors out partway through, or
you close Jupyter and come back later, you won't lose progress or burn
extra rate limit re-fetching repos you already have.

In [10]:
def collect(per_tier_target, out_path, sleep_between=0.1, resume=True):
    rows = []
    completed_tiers = set()
    if resume and os.path.exists(out_path):
        rows = json.load(open(out_path))
        completed_tiers = {r["star_tier"] for r in rows}
        print(f"Resuming: {len(rows)} rows already saved, tiers done: {completed_tiers}")

    for tier_name, qualifier in STAR_TIERS:
        if tier_name in completed_tiers:
            print(f"Skipping tier '{tier_name}' (already done)")
            continue

        print(f"Searching tier '{tier_name}' ({qualifier})...")
        candidates = search_repos(qualifier, per_tier_target)
        print(f"  found {len(candidates)} candidates, fetching detail...")

        for i, c in enumerate(candidates):
            owner, name = c["owner"]["login"], c["name"]
            detail = fetch_repo_detail(owner, name)
            row = flatten(detail)
            if row:
                row["star_tier"] = tier_name
                rows.append(row)
            time.sleep(sleep_between)
            if (i + 1) % 25 == 0:
                print(f"    ...{i+1}/{len(candidates)} in tier '{tier_name}'")

        print(f"  tier '{tier_name}' done, running total rows: {len(rows)}")
        with open(out_path, "w") as f:
            json.dump(rows, f, indent=2, default=str)

    print(f"\nSaved {len(rows)} repos to {out_path}")
    return rows

## Run it

Tip: if you're worried about time or rate limits, first try a small test run
(e.g. `REPOS_PER_TIER = 3`) to make sure everything works end-to-end before
committing to the full 500-repo pull.

In [11]:
rows = collect(REPOS_PER_TIER, OUT_PATH)

Searching tier 'mega' (stars:>50000)...
  found 100 candidates, fetching detail...
    ...25/100 in tier 'mega'
    ...50/100 in tier 'mega'
    ...75/100 in tier 'mega'
    ...100/100 in tier 'mega'
  tier 'mega' done, running total rows: 100
Searching tier 'large' (stars:10000..50000)...
  found 100 candidates, fetching detail...
    ...25/100 in tier 'large'
    ...50/100 in tier 'large'
    ...75/100 in tier 'large'
    ...100/100 in tier 'large'
  tier 'large' done, running total rows: 200
Searching tier 'mid' (stars:1000..10000)...
  found 100 candidates, fetching detail...
    ...25/100 in tier 'mid'
    ...50/100 in tier 'mid'
    ...75/100 in tier 'mid'
    ...100/100 in tier 'mid'
  tier 'mid' done, running total rows: 299
Searching tier 'small' (stars:100..1000)...
  found 100 candidates, fetching detail...
    ...25/100 in tier 'small'
    ...50/100 in tier 'small'
    ...75/100 in tier 'small'
    ...100/100 in tier 'small'
  tier 'small' done, running total rows: 399
Sear

## Quick sanity check on what you collected

In [12]:
import pandas as pd

df = pd.DataFrame(rows)
print("Shape:", df.shape)
print()
print(df["star_tier"].value_counts())
print()
print("Archived:", df["is_archived"].sum())
print("Dormant >365 days:", (df["days_since_last_commit"] > 365).sum())
df.head()

Shape: (499, 21)

star_tier
mega     100
large    100
small    100
tiny     100
mid       99
Name: count, dtype: int64

Archived: 31
Dormant >365 days: 189


,repo,stars,forks,language,license,is_archived,age_days,days_since_last_push,days_since_last_commit,commits_per_month_90d,...,distinct_recent_commit_authors,total_mentionable_users,open_issues_total,closed_issues_total,median_issue_resolution_days_recent,open_prs_total,merged_prs_total,median_pr_merge_days_recent,pr_merge_rate_recent,star_tier
0,codecrafters-io/build-your-own-x,544722,51316,Markdown,NaN,False,3038,49,49.0,1.67,...,38,131,275,673,0.0,365,157,NaN,0.00,mega
1,sindresorhus/awesome,502221,36685,NaN,cc0-1.0,False,4435,0,0.0,2.33,...,68,621,16,352,0.0,89,702,0.0,0.10,mega
2,public-apis/public-apis,474426,52388,Python,mit,False,3817,0,0.0,151.67,...,37,1487,35,911,4.0,1813,2190,1.0,0.32,mega
3,freeCodeCamp/freeCodeCamp,454881,46100,TypeScript,bsd-3-clause,False,4269,0,0.0,269.67,...,39,6226,128,21987,0.0,92,29249,0.0,0.42,mega
4,EbookFoundation/free-programming-books,395797,66724,Python,cc-by-4.0,False,4709,1,1.0,20.00,...,78,3204,39,1258,2.0,44,7431,1.0,0.46,mega
